## MAIN

In [4]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt
import bd_support as sup
from pathlib import Path
from itertools import product

# Picaso
from picaso import justdoit as jdi
from picaso import justplotit as jpi

# Virga
from virga import justdoit as vj
from virga import justplotit as cldplt

# Other
from bokeh.models import Legend
from bokeh.palettes import Category10
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [5]:
# To see what clouds are availible
#vj.available()

In [6]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
opaci_path  = None # Opacity db
# opcai_path  = '/groups/tkaralidi/opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
# output_path = 'home/al864695/ouputs'
pickl_path  = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\bobcat_to_diamondback.pickle"
# pickl_path  = "home/al864695/pickle"

# Things that will remain constant
clouds      = ['Cr','MgSiO3','MnS']
wav_range   = [0.3, 5.0] # microns
MH          = 1.0       # [M/H] metallicity factor ~ solar
MU          = 2.36      # Average MU
R           = 300       # resolution
# res_R       = 5000

# Grid space
Teff_grid = np.linspace(500, 2000, 301)    # length 301, in K
g_grid    = np.linspace(31.0, 1500.0, 41)  # length 41, in m/s^2 (Diamondback min ~31 m/s^2)
fsed_grid = [1, 2, 3, 4, 8]                # length 5
kzz_vals  = np.array([1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]) # cm^2 s^-1

# TP profile setup
corr = sup.TPCorrection(pickl_path)
corr.set_coords(Teff_grid, g_grid, fsed_grid)

In [7]:
##### METHOD BROWN DWARF SPECTRUM

def bd_spectrum(Teff,
                gravity,
                fsed,
                kzz,
                corr=None):
    
    """
    Compute a BD emission spectrum with Virga clouds.
    """

    # Opacity & inputs
    opa = jdi.opannection(wav_range, opaci_path)
    bd = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Inject corrected TP
    _Tcorr = corr.apply_to_picaso_inputs(bd, Teff, gravity, fsed)

    # Inject Kzz (match pressure grid length)
    prof = bd.inputs['atmosphere']['profile']
    P = np.asarray(prof["pressure"], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Clouds
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU)
    out = bd.spectrum(opa, full_output=True)

    # Convert to F_nu, then regrid to constant R in wavenumber
    wn, th = out["wavenumber"], out["thermal"]  # cm^-1 and erg/cm^2/s/cm
    # Convert to F_nu on a wavelength grid first
    sp = jdi.psyn.ArraySpectrum(1e4 / wn, th * 1e-8, waveunits='um', fluxunits='FLAM')

    sp.convert('um')
    sp.convert('Fnu')  # erg/cm^2/s/Hz

    # Regrid at constant resolving power in WAVENUMBER space
    k_in = 1e4 / sp.wave                  # cm^-1
    k_rg, f_rg = jdi.mean_regrid(k_in, sp.flux, R=R)

    # Convert back to wavelength (micron) for plotting
    lam = 1e4 / k_rg                      # um
    idx = np.argsort(lam)                 # ensure ascending wavelength

    lam_um_rg, F_nu_rg = lam[idx], f_rg[idx]

    out['regridx'], out['regridy'] = lam_um_rg, F_nu_rg

    return lam_um_rg, F_nu_rg

In [8]:
##### GENERATE AND SAVE SPECTRUM

# Small test grid
Teff_p = [1500]
grav_p = [100]
fsed_p = [1.0, 2.0]
kzz_p  = [1e9, 2e10]
  
for Teff_i, g_i, fsed_i, kzz_i in product(Teff_p, grav_p, fsed_p, kzz_p):
    lam_i, F_i = bd_spectrum(Teff_i, g_i, fsed_i, kzz_i, corr)

    # Short, readable filename: t<T>g<g>f<fsed><kTag>.npz
    fname = f"t{int(Teff_i)}g{int(g_i)}f{int(fsed_i)}{sup.format_kzz(kzz_i)}.npz"
    fpath = output_path / fname

    # Save
    np.savez_compressed(
        fpath,
        x=np.array([float(Teff_i), float(g_i), float(fsed_i), float(kzz_i)], dtype=float),  # (T, g, fsed, kzz)
        y=F_i.astype(float),                                                                # spectrum
        wavelength_um=lam_i.astype(float)                                                   # wavelength grid
    )

print(f"Saved {len(Teff_p)*len(grav_p)*len(fsed_p)*len(kzz_p)} files to {output_path}")

Saved 4 files to C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs
